# Imports

In [221]:
import random
import torch
import random
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from itertools import product


# Set the default device to CUDA
torch.set_default_device('cuda')

# Create a CUDA random number generator
cuda_generator = torch.Generator(device='cuda')

from sys import exit as e


# Set the default device to CUDA
torch.set_default_device('cuda')

# Verify the default device
print(torch.get_default_device())  # Output: device(type='cuda', index=0)

cuda:2


# Data Generator

In [222]:
def data_generator(input_lengths=[100, 200, 500, 1000], train_size=0.8, val_size=0.2, batch_size=32):
    
    """
    Generate data for the copy task and create PyTorch DataLoaders.
    Sequences of length 1000 are only in the test set.

    Args:
    input_lengths (list): List of sequence lengths to generate.
    train_size (float): Proportion of non-test data for training.
    val_size (float): Proportion of non-test data for validation.
    batch_size (int): Batch size for DataLoaders.

    Returns:
    tuple: (train_loader, val_loader, test_loader, vocab_size, output_size)
    """

    # Define the vocab
    vocabulary = ['N', 'L', 'P', 'I', 'S', 'F', 'U', 'Z']
    delimiter = '##'
    vocab_size = len(vocabulary) + len(delimiter)  # 10 in total

    # Create vocabulary to index mapping 
    char_to_idx = {ch: i for i, ch in enumerate(vocabulary + list(delimiter))}
    
    # Pick a random word from the vocab and generate a sequence of the specified length 
    def generate_sequence(length):
        return ''.join(random.choice(vocabulary) for i in range(length))

    # Convert the generated sequence of letters to sequence of numbers using the char_to_idx dictionary, then convert the generated numbers to onehot encoding
    def sequence_to_tensor(sequence):
        indices = torch.tensor([char_to_idx[ch] for ch in sequence], dtype=torch.long)
        one_hot = torch.nn.functional.one_hot(indices, num_classes=vocab_size)
        return one_hot

    # Creating separate lists for train_val and test because the lengths of 1000 only go in training. 
    train_val_inputs = []
    train_val_targets = []
    test_inputs = []
    test_targets = []

    
    # Iterate in all the lengths, generate 100 samples for each length
    for length in input_lengths:
        for _ in range(100): 
            input_sequence = generate_sequence(length)
            input_with_delimiter = input_sequence + delimiter
            target_sequence = input_sequence

            input_tensor = sequence_to_tensor(input_with_delimiter)
            target_tensor = sequence_to_tensor(target_sequence)

            if length == 1000:
                test_inputs.append(input_tensor)
                test_targets.append(target_tensor)
            else:
                train_val_inputs.append(input_tensor)
                train_val_targets.append(target_tensor)

    # Convert to PyTorch tensors (model architectures are defined using pytorch), pad to account for varying lengths
    train_val_inputs = torch.nn.utils.rnn.pad_sequence(train_val_inputs, batch_first=True)
    train_val_targets = torch.nn.utils.rnn.pad_sequence(train_val_targets, batch_first=True)
    test_inputs = torch.nn.utils.rnn.pad_sequence(test_inputs, batch_first=True)
    test_targets = torch.nn.utils.rnn.pad_sequence(test_targets, batch_first=True)

    # Create train and test datasetsfrom itertools import product

    train_val_dataset = TensorDataset(train_val_inputs, train_val_targets)
    test_dataset = TensorDataset(test_inputs, test_targets)

    # Calculate split sizes for train and validation based in the function parameters
    total_train_val_size = len(train_val_dataset)
    train_size = int(train_size * total_train_val_size)
    val_size = total_train_val_size - train_size

    # Split the train_val dataset
    train_dataset, val_dataset = torch.utils.data.random_split(
        train_val_dataset, [train_size, val_size], generator=cuda_generator
    )

    # Create DataLoaders, this automatically creates batches, making the training easier.
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=cuda_generator)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, generator=cuda_generator)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, generator=cuda_generator)

    # return the generated train, test, and val dataloaders
    return train_loader, val_loader, test_loader, vocab_size




# Model Definitions

## LSTM

In [224]:

# simple lstm without the memory matrix
class StandardLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(StandardLSTM, self).__init__()
        self.hidden_size = hidden_size

        # Define the weight matrices and biases for the LSTM gates
        # nn.Linear layers automatically include bias terms

        # Input gate
        self.W_i = nn.Linear(input_size, hidden_size)  
        self.U_i = nn.Linear(hidden_size, hidden_size)

        # Forget gate
        self.W_f = nn.Linear(input_size, hidden_size)  
        self.U_f = nn.Linear(hidden_size, hidden_size)

        # Output gate
        self.W_o = nn.Linear(input_size, hidden_size)  
        self.U_o = nn.Linear(hidden_size, hidden_size)

        # Cell state
        self.W_c = nn.Linear(input_size, hidden_size) 
        self.U_c = nn.Linear(hidden_size, hidden_size)
        
        # Output layer
        self.output_layer = nn.Linear(hidden_size, output_size)


    def forward(self, x, init_states=None):

        # Ensuring input is float type
        x = x.float()  
        batch_size, seq_length, _ = x.size()
        hidden_seq = []

        # Initialize hidden state and cell state if not provided
        if init_states is None:
            h_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        else:
            h_t, c_t = init_states

        # Process each time step
        for t in range(seq_length):
            x_t = x[:, t, :]
            
            # Calculate gate values
            i_t = torch.sigmoid(self.W_i(x_t) + self.U_i(h_t))
            f_t = torch.sigmoid(self.W_f(x_t) + self.U_f(h_t))
            o_t = torch.sigmoid(self.W_o(x_t) + self.U_o(h_t))
            c_til = torch.tanh(self.W_c(x_t) + self.U_c(h_t))
            
            # Update cell state and hidden state
            c_t = f_t * c_t + i_t * c_til
            h_t = o_t * torch.tanh(c_t)
            
            hidden_seq.append(h_t.unsqueeze(0))

        # Combine all hidden states
        hidden_seq = torch.cat(hidden_seq, dim=0).transpose(0, 1)

        # Generate final output
        output = self.output_layer(hidden_seq)
        return output, output[:, -1, :]


# Multiplicative LSTM (with memomry matrix)
class MultiplicativeLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):

        super(MultiplicativeLSTM, self).__init__()
        self.hidden_size = hidden_size

        # Multiplicative components
        self.W_m = nn.Linear(input_size, input_size)
        self.U_m = nn.Linear(hidden_size, input_size)
        self.b_m = nn.Parameter(torch.zeros(input_size))

        # LSTM components (same as standard)
        self.W_i = nn.Linear(input_size, hidden_size)
        self.U_i = nn.Linear(hidden_size, hidden_size)
        self.W_f = nn.Linear(input_size, hidden_size)
        self.U_f = nn.Linear(hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size, hidden_size)
        self.U_o = nn.Linear(hidden_size, hidden_size)
        self.W_c = nn.Linear(input_size, hidden_size)
        self.U_c = nn.Linear(hidden_size, hidden_size)
        
        # Output layer
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x, init_states=None):
        x = x.float()
        batch_size, seq_length, _ = x.size()
        hidden_seq = []

        if init_states is None:
            h_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
            c_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        else:
            h_t, c_t = init_states

        for t in range(seq_length):
            x_t = x[:, t, :]
            
            # Multiplicative step
            m_t = self.W_m(x_t) + self.U_m(h_t) + self.b_m
            x_til = m_t * x_t
            
            # LSTM gates
            i_t = torch.sigmoid(self.W_i(x_til) + self.U_i(h_t))
            f_t = torch.sigmoid(self.W_f(x_til) + self.U_f(h_t))
            o_t = torch.sigmoid(self.W_o(x_til) + self.U_o(h_t))
            c_tilde = torch.tanh(self.W_c(x_til) + self.U_c(h_t))
            
            # Update states
            c_t = f_t * c_t + i_t * c_tilde
            h_t = o_t * torch.tanh(c_t)
            
            hidden_seq.append(h_t.unsqueeze(0))

        hidden_seq = torch.cat(hidden_seq, dim=0).transpose(0, 1)
        output = self.output_layer(hidden_seq)
        return output, output[:, -1, :]


## GRU

In [225]:
class StandardGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(StandardGRU, self).__init__()
        self.hidden_size = hidden_size

        self.W_z = nn.Linear(input_size, hidden_size)
        self.U_z = nn.Linear(hidden_size, hidden_size)
        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size)
        self.W_h = nn.Linear(input_size, hidden_size)
        self.U_h = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x, init_state=None):
        x = x.float()  # Convert input to float
        batch_size, seq_length, _ = x.size()
        hidden_seq = []

        if init_state is None:
            h_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        else:
            h_t = init_state

        for t in range(seq_length):
            x_t = x[:, t, :]
            
            z_t = torch.sigmoid(self.W_z(x_t) + self.U_z(h_t))
            r_t = torch.sigmoid(self.W_r(x_t) + self.U_r(h_t))
            h_tilde = torch.tanh(self.W_h(x_t) + self.U_h(r_t * h_t))
            h_t = (1 - z_t) * h_t + z_t * h_tilde
            
            hidden_seq.append(h_t.unsqueeze(0))

        hidden_seq = torch.cat(hidden_seq, dim=0).transpose(0, 1)
        output = self.output_layer(hidden_seq)
        return output, output[:, -1, :]

class MultiplicativeGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiplicativeGRU, self).__init__()
        self.hidden_size = hidden_size

        self.W_m = nn.Linear(input_size, input_size)
        self.U_m = nn.Linear(hidden_size, input_size)
        self.b_m = nn.Parameter(torch.zeros(input_size))

        self.W_z = nn.Linear(input_size, hidden_size)
        self.U_z = nn.Linear(hidden_size, hidden_size)
        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size)
        self.W_h = nn.Linear(input_size, hidden_size)
        self.U_h = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x, init_state=None):
        x = x.float()  # Convert input to float
        batch_size, seq_length, _ = x.size()
        hidden_seq = []

        if init_state is None:
            h_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        else:
            h_t = init_state

        for t in range(seq_length):
            x_t = x[:, t, :]
            
            m_t = self.W_m(x_t) + self.U_m(h_t) + self.b_m
            x_tilde = m_t * x_t
            
            z_t = torch.sigmoid(self.W_z(x_tilde) + self.U_z(h_t))
            r_t = torch.sigmoid(self.W_r(x_tilde) + self.U_r(h_t))
            h_tilde = torch.tanh(self.W_h(x_tilde) + self.U_h(r_t * h_t))
            h_t = (1 - z_t) * h_t + z_t * h_tilde
            
            hidden_seq.append(h_t.unsqueeze(0))

        hidden_seq = torch.cat(hidden_seq, dim=0).transpose(0, 1)
        output = self.output_layer(hidden_seq)
        return output, output[:, -1, :]

In [226]:
def calculate_accuracy(outputs, targets):

    # Find the index of the highest probability for each sequence in the batch
    # This represents the model's prediction for each sample
    # For example, if outputs[0] = [0.1, 0.2, 0.5, 0.1, 0.1], predicted[0] would be 2, because the highest probability is 0.5 at index 2
    predicted = outputs.argmax(dim=1)
    
    # Compare the model's predictions to the actual targets
    # The targets are one-hot encoded, so we need to find the index of the 1 in each target
    # For example, if targets[0] = [0, 0, 1, 0, 0], targets.argmax(dim=1)[0] would be 2
    # We then compare this to our predicted value and count how many match
    correct = (predicted == targets.argmax(dim=1)).sum().item()
    
    # Calculate the fraction of correct predictions
    # This is done by dividing the number of correct predictions by the total number of predictions
    # targets.size(0) gives us the number of samples in the batch
    return correct / targets.size(0)


def train_and_evaluate(model, train_loader, val_loader, test_loader, criterion, optimizer, num_epochs, device):
    # Move our model to the specified device (CPU or GPU)
    model.to(device)
    
    # We're going to train for a specified number of epochs
    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()
        train_loss, train_acc = 0.0, 0.0
        
        # Go through each batch in our training data
        for batch_inputs, batch_targets in train_loader:
            # Move our data to the same device as our model
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
            
            # Reset the gradients from the last batch
            optimizer.zero_grad()
            
            # Run our inputs through the model
            # We only care about the last output for each sequence
            _, last_output = model(batch_inputs)
            last_target = batch_targets[:, -1, :]
            
            # Calculate how wrong our predictions are
            loss = criterion(last_output, last_target.float())
            # Calculate what fraction of predictions were correct
            acc = calculate_accuracy(last_output, last_target)
            
            # Figure out how to adjust our model to do better
            loss.backward()
            # Make those adjustments
            optimizer.step()
            
            # Keep track of our total loss and accuracy
            train_loss += loss.item()
            train_acc += acc
        
        # Calculate average loss and accuracy for this epoch
        train_loss /= len(train_loader)
        train_acc /= len(train_loader)

        # Now let's check how we're doing on data we haven't trained on
        model.eval()
        val_loss, val_acc = 0.0, 0.0
        
        # We don't need to calculate gradients for validation
        with torch.no_grad():
            for batch_inputs, batch_targets in val_loader:
                batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
                
                _, last_output = model(batch_inputs)
                last_target = batch_targets[:, -1, :]
                
                loss = criterion(last_output, last_target.float())
                acc = calculate_accuracy(last_output, last_target)
                
                val_loss += loss.item()
                val_acc += acc
        
        # Calculate average validation loss and accuracy
        val_loss /= len(val_loader)
        val_acc /= len(val_loader)
        
        # Print each epoch results
        '''
        print(f'Epoch {epoch+1}/{num_epochs}, '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')'
        '''

    # After all epochs, let's see how we do on the test set
    model.eval()
    test_loss, test_acc = 0.0, 0.0
    with torch.no_grad():
        for batch_inputs, batch_targets in test_loader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
            
            _, last_output = model(batch_inputs)
            last_target = batch_targets[:, -1, :]
            
            loss = criterion(last_output, last_target.float())
            acc = calculate_accuracy(last_output, last_target)
            
            test_loss += loss.item()
            test_acc += acc

    # Calculate and print average test loss and accuracy
    test_loss /= len(test_loader)
    test_acc /= len(test_loader)
    print(f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')

    return test_loss, test_acc



In [ ]:
# Set random seed for reproducibility
torch.manual_seed(000)
np.random.seed(000)

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generate data
train_loader, val_loader, test_loader, vocab_size = data_generator()

# Hyperparameters for grid search
learning_rates = [0.001, 0.0001]
hidden_sizes = [64, 128]
epochs_list = [10, 20]

# Function to perform grid search
def grid_search(model_class, model_name):
    best_acc = 0
    best_params = {}
    results = []

    for lr, hidden_size, num_epochs in product(learning_rates, hidden_sizes, epochs_list):
        print(f"\nTraining {model_name} with lr={lr}, hidden_size={hidden_size}, epochs={num_epochs}")
        
        model = model_class(vocab_size, hidden_size, vocab_size).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        test_loss, test_acc = train_and_evaluate(model, train_loader, val_loader, test_loader, criterion, optimizer, num_epochs, device)
        
        results.append({
            'lr': lr,
            'hidden_size': hidden_size,
            'epochs': num_epochs,
            'test_loss': test_loss,
            'test_acc': test_acc
        })
        
        if test_acc > best_acc:
            best_acc = test_acc
            best_params = {'lr': lr, 'hidden_size': hidden_size, 'epochs': num_epochs}
    
    return results, best_params

# Perform grid search for each model
models = [
    (StandardLSTM, "StandardLSTM"),
    (MultiplicativeLSTM, "MultiplicativeLSTM"),
    (StandardGRU, "StandardGRU"),
    (MultiplicativeGRU, "MultiplicativeGRU")
]

for model_class, model_name in models:
    print(f"\nPerforming grid search for {model_name}")
    results, best_params = grid_search(model_class, model_name)
    
    print(f"\nResults for {model_name}:")
    for result in results:
        print(f"lr={result['lr']}, hidden_size={result['hidden_size']}, epochs={result['epochs']}")
        print(f"Test Loss: {result['test_loss']:.4f}, Test Acc: {result['test_acc']:.4f}")
    
    print(f"\nBest parameters for {model_name}:")
    print(f"lr={best_params['lr']}, hidden_size={best_params['hidden_size']}, epochs={best_params['epochs']}")
    print(f"Best Test Acc: {max(result['test_acc'] for result in results):.4f}")
